# Structured Outputs in LangChain

In LangChain, obtaining structured outputs from language models is a crucial design pattern for building robust applications. Instead of raw text, models can return fully parsed, type-safe data structures.

LangChain supports two primary ways of defining these schemas:
1. **Pydantic** (for robust classes, runtime validation, and coercion)
2. **TypedDict** (for lightweight, native Python dictionary type-hinting)

## 1. Structured Outputs with Pydantic

### Core Concepts:

- **What is Pydantic?**
  - **Pydantic** is Python's most popular data validation library.
  - By defining a Pydantic model (`BaseModel`), you specify attributes, types, and optional descriptions (`Field`). LangChain uses these to instruct the LLM.

- **The `.with_structured_output()` Method**:
  - This is the unified interface in LangChain for structured output. It leverages model-specific capabilities such as **Tool/Function Calling** or **JSON Mode**.

- **Key Benefits**:
  - **Type Safety & Coercion**: Guarantees types and coerces them (e.g. string "2010" into integer `2010`).
  - **Automatic Parsing**: Instantiates a real Python class directly.

In [1]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

# Load API keys
load_dotenv()

# Initialize our chat model
model = init_chat_model("google_genai:gemini-2.5-flash-lite")

### Defining our Pydantic Schema

Here we define our desired output structure using Pydantic.

In [2]:
from pydantic import BaseModel, Field
from typing import List, Optional

class Actor(BaseModel):
    name: str = Field(description="The name of the actor")
    role: str = Field(description="The character role they played in the movie")

class MovieDetails(BaseModel):
    title: str = Field(description="The title of the movie")
    release_year: int = Field(description="The year the movie was released")
    director: str = Field(description="The director of the movie")
    genres: List[str] = Field(description="List of genres for the movie")
    cast: List[Actor] = Field(description="List of main actors and their roles")
    rating: Optional[float] = Field(None, description="The IMDb or critic rating out of 10, if known")

### Invoking the Model with Pydantic

We bind our `MovieDetails` schema to the model using `.with_structured_output()`. When invoked, the returned response is a Pydantic object.

In [3]:
# Bind the schema to create a structured runnable
structured_model = model.with_structured_output(MovieDetails)

# Run extraction query
query = """
Extract details for the movie bahubali
"""
movie = structured_model.invoke(query)

# The result is a clean, typed Pydantic object!
print(f"Type of output: {type(movie)}")
print(f"Movie: {movie.title} ({movie.release_year})")
print(f"Director: {movie.director}")
print(f"Rating: {movie.rating}")
print(f"Genres: {movie.genres}")
print("Cast:")
for actor in movie.cast:
    print(f" - {actor.name} as {actor.role}")

ConnectError: [Errno -3] Temporary failure in name resolution

## 2. Structured Outputs with TypedDict

### Core Concepts:

- **What is TypedDict?**
  - Introduced in Python 3.8, `TypedDict` allows you to declare type hints for dictionaries with a fixed set of keys and specific value types.

- **Pydantic vs. TypedDict**:
  - **Pydantic**: Returns a custom Pydantic object, performs strict runtime validation, and coerces types automatically.
  - **TypedDict**: Returns a standard Python dictionary (`dict`). There is no runtime validation overhead; it is purely a type-hinting mechanism for your IDE and static analysis tools.

- **When to use TypedDict**:
  - If you want a lightweight solution that avoids introducing Pydantic dependencies or validation steps.
  - When working with libraries that expect raw dictionaries (such as state management in **LangGraph**).

### Defining our TypedDict Schema

In [ ]:
from typing import TypedDict, List

class ActorDict(TypedDict):
    name: str
    role: str

class MovieDetailsDict(TypedDict):
    title: str
    release_year: int
    director: str
    genres: List[str]
    cast: List[ActorDict]

### Invoking the Model with TypedDict

Just like with Pydantic, we pass our `TypedDict` class directly to `.with_structured_output()`. The model returns a standard Python dictionary matching the schema.

In [ ]:
# Bind the TypedDict schema to create a structured runnable
structured_dict_model = model.with_structured_output(MovieDetailsDict)

# Run extraction query
query = """
Extract details for the movie avathar
"""
movie_dict = structured_dict_model.invoke(query)

# The result is a standard Python dictionary!
print(f"Type of output: {type(movie_dict)}")
print(movie_dict)

# Accessing dictionary elements
print(f"\nMovie: {movie_dict.get('title')} ({movie_dict.get('release_year')})")
print(f"Director: {movie_dict.get('director')}")
print(f"Genres: {movie_dict.get('genres')}")
print("Cast:")
for actor in movie_dict.get('cast', []):
    print(f" - {actor.get('name')} as {actor.get('role')}")